In [ ]:
#========================================================================
# Name: calc_wrf_properties_lasso.ipynb
# Author: McKenna W. Stanford
# Author Contact: mckenna.stanford@pnnl.gov
#
# Description: Extract WRF microphysical properties and compute cloud-top
#              parameters using Mie scattering for LASSO 2.5km domain.
#========================================================================

In [ ]:
#===============================
# Imports
#===============================
import xarray as xr
import numpy as np
import glob
import datetime
import scipy.ndimage as ndimage
import time
import os
import dask
from dask.distributed import wait
from distributed import Client, LocalCluster
from functions import find_nearest, to_datetime
import pickle
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.spatial import cKDTree
from scipy.ndimage import label
import miepython
from scipy.interpolate import interp1d


import warnings
warnings.filterwarnings("ignore")

In [ ]:
iparallel = True

# Start Dask client

In [ ]:
if iparallel:
    dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 
    cluster = LocalCluster(n_workers=32,threads_per_worker=1)
    client = Client(cluster)

In [ ]:
client

In [ ]:
sim_date = '20181129'
domain = 'd4'
ens = 'gefs03'

path = f'/pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/{sim_date}/{ens}/{domain}/cld_met_files/'

In [ ]:
cld_files = sorted(glob.glob(path+'*cldhamsl*.nc'))[::3]
met_files = sorted(glob.glob(path+'*methamsl*.nc'))[::3]
num_cld_files = len(cld_files)
num_met_files = len(met_files)
print('# of cld files:',num_cld_files)
print('# of met files:',num_met_files)

In [ ]:
cld_files

In [ ]:
cld_datetimes = []
for ii in range(num_cld_files):
    #print(cld_files[ii])
    if domain != 'd2':
        dum_str = cld_files[ii].split('/')[-1].split('_')[-1].split('.')
        date_str = dum_str[2]
        time_str = dum_str[3]
        cld_datetimes.append(datetime.datetime(int(date_str[0:4]),int(date_str[4:6]),int(date_str[6:8]),int(time_str[0:2]),int(time_str[2:4]),0,tzinfo=datetime.timezone.utc))
    else:
        dum_str = cld_files[ii].split('/')[-1].split('.')
        date_str = dum_str[-3]
        time_str = dum_str[-2]
        cld_datetimes.append(datetime.datetime(int(date_str[0:4]),int(date_str[4:6]),int(date_str[6:8]),int(time_str[0:2]),int(time_str[2:4]),0,tzinfo=datetime.timezone.utc))

In [ ]:
#cld_datetimes

# Compute LUTs for optical efficiencies

In [ ]:
# ===========================
# Lookup-table Mie computation
# ===========================
def compute_Q_vectorized(m, radii, wavelength):
    """
    Vectorized computation of extinction, scattering, absorption efficiencies
    for an array of particle radii using miepython.
    """
    diameters = 2.0 * radii
    qext, qsca, qback, g = miepython.efficiencies(m, diameters, wavelength)
    qabs = qext - qsca
    return qext, qsca, qabs

def maxwell_garnett(m_incl, m_host, f_incl):
    """
    Maxwell-Garnett effective refractive index for inclusions in host.
    """
    eps_incl = m_incl**2
    eps_host = m_host**2
    num = eps_incl + 2*eps_host + 2*f_incl*(eps_incl - eps_host)
    den = eps_incl + 2*eps_host - f_incl*(eps_incl - eps_host)
    eps_eff = eps_host * (num / den)
    return np.sqrt(eps_eff)

In [ ]:
# -------------------------------------------------------
# Optical properties from literature
# -------------------------------------------------------
wl_VIS = 0.64    #  (visible)
wl_IR  = 11.2    # (infrared)

# Refractive indices
m_liq_VIS = 1.331 + 1.6e-8j
m_liq_IR  = 1.153 + 0.0968j
m_ice_VIS = 1.3083 + 1.22e-8j
m_ice_IR  = 1.0925 + 0.248j

rho_ice = 890.0
rho_snow = 100.0
f_ice = rho_snow / rho_ice

m_snow_VIS = maxwell_garnett(m_ice_VIS, 1.0+0j, f_ice)
m_snow_IR  = maxwell_garnett(m_ice_IR, 1.0+0j, f_ice)

# Effective radius lookup values (1 μm to 500 μm)
radii_lut = np.logspace(-6, -3, 100)*1.e6  # meters

# Precompute LUTs
print("Computing Mie LUTs...")
start_time = time.time()
lut = {}
for name, m, wl in [
    ("cloud_VIS", m_liq_VIS, wl_VIS),
    ("cloud_IR", m_liq_IR, wl_IR),
    ("ice_VIS", m_ice_VIS, wl_VIS),
    ("ice_IR", m_ice_IR, wl_IR),
    ("snow_VIS", m_snow_VIS, wl_VIS),
    ("snow_IR", m_snow_IR, wl_IR)
]:
    qext, qsca, qabs = compute_Q_vectorized(m, radii_lut, wl)
    lut[name] = {"qext": qext, "qsca": qsca, "qabs": qabs}

# Override VIS: all scattering = 2, absorption = 0
for hydro in ['cloud', 'ice', 'snow']:
    lut[f"{hydro}_VIS"]['qsca'][:] = 2.0
    lut[f"{hydro}_VIS"]['qabs'][:] = 0.0
    lut['radii'] = radii_lut

end_time = time.time()
print(f"LUT computation complete in {end_time-start_time:.2f} s")

# Helper Functions

## Subset 2D variables within radar domain

In [ ]:
def subset_var(data,mask):

    # mask values where the csapr mask == 0.
    masked_data = np.ma.masked_array(data, mask=mask == 0)

    # Extract the 2D subset defined by the mask
    # The unmasked elements will define the dimensions
    unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data


    row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices

    new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
    new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region

    reshaped_data = unmasked_data.reshape((new_rows, new_cols))

    return reshaped_data.data

## Subset 3D variables within radar domain

In [ ]:
def subset_var_3d(data,mask):

    nz = len(data[:,0,0])
    # mask values where the csapr mask == 0.
    reshaped_data_3d = []
    
    for kk in range(nz):
        data_2d = data[kk,:,:]
        masked_data = np.ma.masked_array(data_2d, mask=mask == 0)
        #print(aaaaa)
        
        # Extract the 2D subset defined by the mask
        # The unmasked elements will define the dimensions
        unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data
        
        
        row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices
        
        new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
        new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region
        
        reshaped_data = unmasked_data.reshape((new_rows, new_cols))
    
        reshaped_data_3d.append(reshaped_data.data)

    reshaped_data_3d = np.asarray(reshaped_data_3d)
    
    return reshaped_data_3d

## Get the radar mask to NaN out points outside fo the 150-km radial range ring

In [ ]:
csapr_path = '/pscratch/sd/m/mckenna/cacti/csapr_regridded/'
csapr_files = sorted(glob.glob(csapr_path+'/*.nc'))
#ds = xr.open_dataset(csapr_files[0])

In [ ]:
def get_csapr_mask(csapr_file, cld_file):
    """
    Creates a rectangular mask for the CLD grid based on the CSAPR domain.
    The final mask is a perfect rectangle in grid-index space.
    """
    with xr.open_dataset(csapr_file) as ds_csapr:
        lon_csapr = ds_csapr['point_longitude'].squeeze()
        lat_csapr = ds_csapr['point_latitude'].squeeze()

    with xr.open_dataset(cld_file) as ds_cld:
        lon_cld = ds_cld['XLONG'].squeeze()
        lat_cld = ds_cld['XLAT'].squeeze()

    # --- Step 1: Create the initial geographic mask (as before) ---
    lat_min_csapr = lat_csapr.min().item()
    lat_max_csapr = lat_csapr.max().item()
    lon_min_csapr = lon_csapr.min().item()
    lon_max_csapr = lon_csapr.max().item()

    geographic_mask = (
        (lon_cld >= lon_min_csapr) & (lon_cld <= lon_max_csapr) &
        (lat_cld >= lat_min_csapr) & (lat_cld <= lat_max_csapr)
    ).values  # Work with numpy array from here

    # --- Step 2: Find the min/max row and column indices of the mask ---
    # .any(axis=0) -> checks which COLUMNS have at least one 'True'
    # .any(axis=1) -> checks which ROWS have at least one 'True'
    
    # Find the first and last column index that contain a valid point
    cols_with_data = np.where(geographic_mask.any(axis=0))[0]
    if len(cols_with_data) == 0: # Handle case where there's no overlap
        return np.zeros_like(lon_cld.values, dtype=float)
    j_min, j_max = cols_with_data.min(), cols_with_data.max()

    # Find the first and last row index that contain a valid point
    rows_with_data = np.where(geographic_mask.any(axis=1))[0]
    i_min, i_max = rows_with_data.min(), rows_with_data.max()

    # --- Step 3 & 4: Create a new mask and fill the rectangular region ---
    rectangular_mask = np.zeros_like(geographic_mask, dtype=float)
    # Use slicing to fill the bounding box in index space
    rectangular_mask[i_min : i_max + 1, j_min : j_max + 1] = 1.0
    
    # Return the final rectangular mask and the corner indices for visualization
    corner_indices = {'i_min': i_min, 'i_max': i_max, 'j_min': j_min, 'j_max': j_max}
    #return rectangular_mask, lon_cld.values, lat_cld.values, corner_indices
    return rectangular_mask

## Compute column-maximum reflectivity

In [ ]:
def compute_col_max_ref(cld_file,csapr_file):
    ds_cld = xr.open_dataset(cld_file)
    ref = ds_cld['REFL_10CM'].squeeze()
    ref_coords = ref.coords
    lon = ds_cld['XLONG']
    lat = ds_cld['XLAT']
    zamsl = ds_cld['HAMSL'] # 1D
    ds_cld.close()


    # Use xarray.broadcast to expand zamsl to 3D 
    # This aligns zamsl with ref based on common dimension names 
    # and broadcasts it to match ref's shape.
    zamsl_3d, _ = xr.broadcast(zamsl, ref)
    # The result zamsl_3d will have the shape (40, 111, 86)
    # ------------------------------------------------------------
    
    # Compute column max reflectivity
    col_max_ref = np.nanmax(ref, axis=0)
    # Mask where col_max_ref is <= -35
    valid_mask = col_max_ref > -35

    # Initialize col_max_ref_arg with NaNs
    col_max_ref_arg = np.full(ref.shape[1:], np.nan, dtype=float)

    if np.any(valid_mask):  # Only compute where there are valid values
        # Replace NaNs with -inf to prevent failures in nanargmax
        ref_filled = np.where(np.isnan(ref), -np.inf, ref)

        # Compute indices safely
        valid_indices = np.nanargmax(ref_filled, axis=0)

        # Assign indices only where col_max_ref > -35
        col_max_ref_arg[valid_mask] = valid_indices[valid_mask]
        
    csapr_mask = get_csapr_mask(csapr_file,cld_file)
    col_max_ref = subset_var(col_max_ref,csapr_mask)
    col_max_ref_arg = subset_var(col_max_ref_arg,csapr_mask)
    lon = subset_var(lon,csapr_mask)
    lat = subset_var(lat,csapr_mask)
    zamsl_3d = subset_var_3d(zamsl_3d,csapr_mask)

    ref_dict = {'col_max_ref':col_max_ref,
                'col_max_ref_arg':col_max_ref_arg,
                'lat':lat,
                'lon':lon,
                'z':zamsl_3d,
               }
    return ref_dict


In [ ]:
ds = xr.open_dataset(csapr_files[200])
col_max_ref = np.nanmax(ds['taranis_attenuation_corrected_reflectivity'].values.squeeze(),axis=0)
plt.contourf(col_max_ref)


## Grab variables from 2D files

In [ ]:
def get_2d_vars(cld_file,csapr_file):

    ds_cld = xr.open_dataset(cld_file)
    iwp = ds_cld['IWP'].squeeze()
    lwp = ds_cld['LWP'].squeeze()
    ds_cld.close()

    csapr_mask = get_csapr_mask(csapr_file,cld_file)
    iwp = subset_var(iwp,csapr_mask)
    lwp = subset_var(lwp,csapr_mask)
    
    twp = lwp + iwp
    
    
    x2d_dict = {'lwp':lwp,
                'iwp':iwp,
                'twp':twp,
               }
    
    return x2d_dict

## Calculate CTT using condensate threshold

In [ ]:
def calc_ctt(temp,height,qt,cloud_mask,nx_2d,ny_2d):

    ctt = np.zeros((ny_2d,nx_2d))-999.
    cth = np.zeros((ny_2d,nx_2d))-999.

    for iii in range(nx_2d):
        for jjj in range(ny_2d):
            sc_temp = np.flip(temp[:,jjj,iii])
            sc_height = np.flip(height[:,jjj,iii])
            sc_qt = np.flip(qt[:,jjj,iii])
            sc_cloud_mask = np.flip(cloud_mask[:,jjj,iii])
            orig_sc_cloud_mask = np.flip(cloud_mask[:,jjj,iii])
            dumid = np.where(sc_cloud_mask > 0.)
            
            if (np.max(sc_cloud_mask) == 0.) or (np.size(dumid) <= 2.):
                continue
            else:
                labeled_array, num_features = ndimage.label(sc_cloud_mask)
                unique_ids = np.unique(labeled_array)[1:]
                
                for kk in range(num_features):
                    dumid = np.where(labeled_array == unique_ids[kk])
                    tmp_segment_size = np.size(dumid)
                    if tmp_segment_size == 1.:
                        dumid = dumid[0][0]
                        sc_cloud_mask[labeled_array == unique_ids[kk]] = 0
                    
                labeled_array, num_features = ndimage.label(sc_cloud_mask)
                if num_features > 0.:
                    unique_ids = np.unique(labeled_array)[1:]
                    ct_id = np.where(labeled_array > 0)[0][0]
                    ctt[jjj,iii] = sc_temp[ct_id]
                    cth[jjj,iii] = sc_height[ct_id]
                else:
                    pass

    ctt[ctt == -999.] = np.nan
    cth[cth == -999.] = np.nan

    return ctt,cth

## Calculate effective radius in a manner faithful to Thompson AA microphysics scheme

In [ ]:
def calc_effectRad(t1d, p1d, qv1d, qc1d, nc1d, qi1d, ni1d, qs1d, 
                   re_qc1d, re_qi1d, re_qs1d, kts, kte, is_aerosol_aware=True):

    # Constants
    R1 = 1.E-12
    R2 = 1.E-6
    rho_w = 1000.0
    rho_s = 100.0
    rho_i = 890.0
    am_r = np.pi * rho_w / 6.0
    am_i = np.pi * rho_i / 6.0
    am_s = 0.069
    bm_r = 3.0
    bm_i = 3.0
    bm_s = 2.0
    oams = 1./am_s
    bv_s = 0.55
    mu_s = 0.635
    
    # Sa and Sb arrays (for snow moments conversions)
    sa = np.array([5.065339, -0.062659, -3.032362, 0.029469, -0.000285, 
                   0.31255,   0.000204,  0.003199, 0.0,      -0.015952])
    sb = np.array([0.476221, -0.015896,  0.165977, 0.007468, -0.000141, 
                   0.060366,  0.000079,  0.000594, 0.0,      -0.003577])
    
    # G ratio for interpolation
    g_ratio = np.array([24,60,120,210,336,504,720,990,1320,1716,2184,2730,3360,4080,4896])
    
    # Define cse array based on the original Fortran code
    cse = np.zeros(17)
    cse[0] = bm_s + 1.0
    cse[1] = bm_s + 2.0
    cse[2] = bm_s * 2.0
    cse[3] = bm_s + bv_s + 1.0
    cse[4] = bm_s * 2.0 + bv_s + 1.0
    cse[5] = bm_s * 2.0 + 1.0
    cse[6] = bm_s + mu_s + 1.0
    cse[7] = bm_s + mu_s + 2.0
    cse[8] = bm_s + mu_s + 3.0
    cse[9] = bm_s + mu_s + bv_s + 1.0
    cse[10] = bm_s * 2.0 + mu_s + bv_s + 1.0
    cse[11] = bm_s * 2.0 + mu_s + 1.0
    cse[12] = bv_s + 2.0
    cse[13] = bm_s + bv_s
    cse[14] = mu_s + 1.0
    cse[15] = 1.0 + (1.0 + bv_s) / 2.0
    cse[16] = bm_s + bv_s + 2.0


    
    # Initializations
    rho = np.zeros_like(t1d)
    rc = np.zeros_like(t1d)
    nc = np.zeros_like(t1d)
    ri = np.zeros_like(t1d)
    ni = np.zeros_like(t1d)
    rs = np.zeros_like(t1d)
    
    has_qc = False
    has_qi = False
    has_qs = False

    re_qc1d[:] = 2.51E-6  # Default values
    re_qi1d[:] = 2.51E-6  # Default values
    re_qs1d[:] = 5.01E-6  # Default values

    # Compute radii and check conditions
    for k in range(kts-1, kte):  # Python indexing is 0-based, so adjust indices
        rho[k] = 0.622 * p1d[k] / (287.04 * t1d[k] * (qv1d[k] + 0.622))  # R is 287.04 J/(kg*K)
        rc[k] = max(R1, qc1d[k] * rho[k])
        nc[k] = max(2.0, min(nc1d[k] * rho[k], 1000.0))
        if not is_aerosol_aware:
            nc[k] = 200.0  # Default value for non-aerosol-aware
        if rc[k] > R1 and nc[k] > R2:
            has_qc = True

        ri[k] = max(R1, qi1d[k] * rho[k])
        ni[k] = max(R2, ni1d[k] * rho[k])
        if ri[k] > R1 and ni[k] > R2:
            has_qi = True

        rs[k] = max(R1, qs1d[k] * rho[k])
        if rs[k] > R1:
            has_qs = True

    # Update effective radii based on conditions
    if has_qc:
        for k in range(kts-1, kte):
            if rc[k] <= R1 or nc[k] <= R2:
                re_qc1d[k] = np.nan
                continue
            inu_c = 15 if nc[k] < 100 else min(15, int(1000.0 / nc[k]) + 2)
            lamc = (nc[k] * am_r * g_ratio[inu_c-1] / rc[k])**(1.0/bm_r)
            re_qc1d[k] = max(2.51E-6, min(0.5 * (3.0 + inu_c) / lamc, 50.E-6))
    else:
        re_qc1d[:] = np.nan

    if has_qi:
        for k in range(kts-1, kte):
            if ri[k] <= R1 or ni[k] <= R2:
                re_qi1d[k] = np.nan
                continue
            lami = (am_i * 2.0 * ni[k] / ri[k])**(1.0/bm_i)
            re_qi1d[k] = max(2.51E-6, min(0.5 * (3.0 + 2.0) / lami, 125.E-6))
    else:
        re_qi1d[:] = np.nan

    # Compute re_qs1d if has_qs
    if has_qs:
        for k in range(kts-1, kte):  # Python indexing is 0-based, adjust accordingly
            if rs[k] <= R1:
                re_qs1d[k] = np.nan
                continue
            tc0 = min(-0.1, t1d[k] - 273.15)
            smob = rs[k] * oams

            if abs(bm_s - 2.0) < 1.e-3:
                smo2 = smob
            else:
                loga_ = sa[0] + sa[1] * tc0 + sa[2] * bm_s + sa[3] * tc0 * bm_s \
                        + sa[4] * tc0 ** 2 + sa[5] * bm_s ** 2 + sa[6] * tc0 ** 2 * bm_s \
                        + sa[7] * tc0 * bm_s ** 2 + sa[8] * tc0 ** 3 + sa[9] * bm_s ** 3
                a_ = 10.0 ** loga_
                b_ = sb[0] + sb[1] * tc0 + sb[2] * bm_s + sb[3] * tc0 * bm_s \
                        + sb[4] * tc0 ** 2 + sb[5] * bm_s ** 2 + sb[6] * tc0 ** 2 * bm_s \
                        + sb[7] * tc0 * bm_s ** 2 + sb[8] * tc0 ** 3 + sb[9] * bm_s ** 3
                smo2 = (smob / a_) ** (1. / b_)

            # Second part of the computation (using cse)
            loga_ = sa[0] + sa[1] * tc0 + sa[2] * cse[0] + sa[3] * tc0 * cse[0] \
                    + sa[4] * tc0 ** 2 + sa[5] * cse[0] ** 2 + sa[6] * tc0 ** 2 * cse[0] \
                    + sa[7] * tc0 * cse[0] ** 2 + sa[8] * tc0 ** 3 + sa[9] * cse[0] ** 3
            a_ = 10.0 ** loga_
            b_ = sb[0] + sb[1] * tc0 + sb[2] * cse[0] + sb[3] * tc0 * cse[0] \
                    + sb[4] * tc0 ** 2 + sb[5] * cse[0] ** 2 + sb[6] * tc0 ** 2 * cse[0] \
                    + sb[7] * tc0 * cse[0] ** 2 + sb[8] * tc0 ** 3 + sb[9] * cse[0] ** 3
            smoc = a_ * smo2 ** b_
            re_qs1d[k] = max(5.01E-6, min(0.5 * (smoc / smob), 999.E-6))
    else:
        re_qs1d[:] = np.nan


    # Example usage
    # You would call calc_effectRad with the proper input arrays and indices.
    return re_qc1d, re_qi1d, re_qs1d

# Compute layer thickness

In [ ]:
def compute_dz(height):
    """Compute layer thickness (dz) from height midpoints."""
    height_mid = 0.5 * (height[:-1,:,:] + height[1:,:,:])
    dz = height_mid[1:,:,:] - height_mid[:-1,:,:]
    return dz

# Interpolate efficiencies from LUT to the provided Reff

In [ ]:
def interp_Q(LUT, re_eff, hydro, wl):
    """Interpolate Q_scat and Q_abs from LUT for each grid point."""
    key = f"{hydro}_{wl}"
    qsca_lut = LUT[key]["qsca"]
    qabs_lut = LUT[key]["qabs"]
    radii_lut = LUT["radii"]
    # flatten for interpolation
    shape = re_eff.shape
    re_flat = re_eff.flatten()
    qsca = interp1d(radii_lut, qsca_lut, bounds_error=False, fill_value=(qsca_lut[0], qsca_lut[-1]))(re_flat)
    qabs = interp1d(radii_lut, qabs_lut, bounds_error=False, fill_value=(qabs_lut[0], qabs_lut[-1]))(re_flat)
    return qsca.reshape(shape), qabs.reshape(shape)

# Compute optical depth per hydrometeor

In [ ]:
def compute_tau(q, r_eff, rho_hydro, dz, Q_scat, Q_abs):
    """Vectorized optical depth computation."""
    mask = ~np.isnan(r_eff)
    tau_scat = np.zeros_like(dz)
    tau_abs = np.zeros_like(dz)
    tau_scat[mask] = (3 * Q_scat[mask] * q[mask]) / (2 * rho_hydro * r_eff[mask]) * dz[mask]
    tau_abs[mask] = (3 * Q_abs[mask] * q[mask]) / (2 * rho_hydro * r_eff[mask]) * dz[mask]
    return tau_scat + tau_abs

# Compute optical depth considering all hydrometeors

In [ ]:
def compute_all_tau(q_dict, r_eff_dict, dz, DENSITIES, LUT, weightings):
    """
    Compute optical depths for VIS, IR, and weighted combinations.
    """
    tau_VIS = np.zeros_like(dz)
    tau_IR = np.zeros_like(dz)
    
    for hydro in q_dict.keys():
        rho = DENSITIES[hydro]
        q = q_dict[hydro]
        r_eff = r_eff_dict[hydro]
        Q_scat_VIS, Q_abs_VIS = interp_Q(LUT, r_eff, hydro, "VIS")
        Q_scat_IR,  Q_abs_IR  = interp_Q(LUT, r_eff, hydro, "IR")
        tau_VIS += compute_tau(q, r_eff, rho, dz, Q_scat_VIS, Q_abs_VIS)
        tau_IR  += compute_tau(q, r_eff, rho, dz, Q_scat_IR, Q_abs_IR)
    
    tau_combined = {}
    for key, (w_VIS, w_IR) in weightings.items():
        tau_combined[key] = w_VIS*tau_VIS + w_IR*tau_IR
    
    return tau_VIS, tau_IR, tau_combined

# Compute CTT/CTH from cumulative optical depth (vectorized)

In [ ]:
# ===========================
# Vectorized CTT/CTH from cumulative tau
# ===========================
def compute_ctt_cth_vectorized(tau_dict, temperature, height, tau_threshold=1.0):
    nz, ny, nx = temperature.shape
    temp_rev = temperature[::-1,:,:]
    height_rev = height[::-1,:,:]

    results = {}

    # VIS and IR
    for key in ['tau_VIS', 'tau_IR']:
        tau_rev = tau_dict[key][::-1,:,:]
        cum_tau = np.nancumsum(tau_rev, axis=0)
        exceed_idx = np.argmax(cum_tau > tau_threshold, axis=0)
        valid_mask = np.any(cum_tau > tau_threshold, axis=0)
        ctt = np.full((ny,nx), np.nan)
        cth = np.full((ny,nx), np.nan)
        rows, cols = np.where(valid_mask)
        ctt[rows,cols] = temp_rev[exceed_idx[rows,cols], rows, cols]
        cth[rows,cols] = height_rev[exceed_idx[rows,cols], rows, cols]
        results[f'ctt_{key}'] = ctt-273.15
        results[f'cth_{key}'] = cth*1.e-3

    # Weighted combinations
    for wkey, tau_comb in tau_dict['tau_combined'].items():
        tau_rev = tau_comb[::-1,:,:]
        cum_tau = np.nancumsum(tau_rev, axis=0)
        exceed_idx = np.argmax(cum_tau > tau_threshold, axis=0)
        valid_mask = np.any(cum_tau > tau_threshold, axis=0)
        ctt = np.full((ny,nx), np.nan)
        cth = np.full((ny,nx), np.nan)
        rows, cols = np.where(valid_mask)
        ctt[rows,cols] = temp_rev[exceed_idx[rows,cols], rows, cols]
        cth[rows,cols] = height_rev[exceed_idx[rows,cols], rows, cols]
        results[f'ctt_tau_{wkey}'] = ctt-273.15
        results[f'cth_tau_{wkey}'] = cth*1.e-3

    return results

In [ ]:
def get_3d_vars(cld_file,met_file,csapr_file):
    #=======================
    # Load 3D variables
    #=======================
    ds_cld = xr.open_dataset(cld_file)
    ds_met = xr.open_dataset(met_file)

    temp = ds_met['TEMPERATURE'].values.squeeze()-273.15 # deg C
    pressure = ds_met['PRESSURE'].values.squeeze()*1.e2 # Pa
    qv = ds_met['QVAPOR'].values.squeeze()*1.e3 # g/kg
    #zamsl_1d = ds_met['HAMSL'].values.squeeze() # meters

    # Use xarray.broadcast to expand zamsl to 3D 
    # This aligns zamsl with ref based on common dimension names 
    # and broadcasts it to match ref's shape.
    z, _ = xr.broadcast(ds_met['HAMSL'], ds_met['TEMPERATURE'])
    z = z.values.squeeze()
    # The result zamsl_3d will have the shape (40, 111, 86)
    # ------------------------------------------------------------
    
    qc = ds_cld['QCLOUD'].values.squeeze()*1.e3 # g/kg
    qi = ds_cld['QICE'].values.squeeze()*1.e3 # g/kg
    qs = ds_cld['QSNOW'].values.squeeze()*1.e3 # g/kg
    qr = ds_cld['QRAIN'].values.squeeze()*1.e3 # g/kg 
    qg = ds_cld['QGRAUP'].values.squeeze()*1.e3 # g/kg
    nc = ds_cld['QNCLOUD'].values.squeeze() # /kg
    ni = ds_cld['QNICE'].values.squeeze() # /kg
    nr = ds_cld['QNRAIN'].values.squeeze() # /kg
    
    ds_cld.close()
    ds_met.close()
    
    csapr_mask = get_csapr_mask(csapr_file,cld_file)
    temp = subset_var_3d(temp,csapr_mask)
    pressure = subset_var_3d(pressure,csapr_mask)
    qv = subset_var_3d(qv,csapr_mask)
    z = subset_var_3d(z,csapr_mask)
    qc = subset_var_3d(qc,csapr_mask)
    qi = subset_var_3d(qi,csapr_mask)
    qs = subset_var_3d(qs,csapr_mask)
    qr = subset_var_3d(qr,csapr_mask)
    qg = subset_var_3d(qg,csapr_mask)
    nc = subset_var_3d(nc,csapr_mask)
    ni = subset_var_3d(ni,csapr_mask)
    nr = subset_var_3d(nr,csapr_mask)

    
    #=======================
    # Air density
    #=======================
    Rd=287.04
    Tv = (temp+237.15)*(1. + 0.61*qv)
    rho_air = pressure/(Rd*Tv)

    qt = qc + qi + qs # g/kg    
    nz,ny,nx = temp.shape
    
    print('Computing CTT using condensate thresholds...')
    #--------------------------
    # Thresh 1 (Lenient)
    #--------------------------
    thresh = 1.e-6
    cloud_id = np.where(qt > thresh)
    
    if np.size(cloud_id) > 0.:
        cloud_mask = np.zeros([nz,ny,nx])
        cloud_mask[cloud_id] = 1.
        ctt_thresh_1,cth_thresh_1 = calc_ctt(temp,z,qt,cloud_mask,nx,ny)
    else:
        ctt_thresh_1 = np.zeros((ny,nx))
        cth_thresh_1 = np.zeros((ny,nx))
        ctt_thresh_1[:,:] = np.nan
        cth_thresh_1[:,:] = np.nan
    
    #--------------------------
    # Thresh 2 (Generous)
    #--------------------------
    thresh = 1.e-3
    cloud_id = np.where(qt > thresh)
    
    if np.size(cloud_id) > 0.:
        cloud_mask = np.zeros([nz,ny,nx])
        cloud_mask[cloud_id] = 1.
        ctt_thresh_2,cth_thresh_2 = calc_ctt(temp,z,qt,cloud_mask,nx,ny)
    else:
        ctt_thresh_2 = np.zeros((ny,nx))
        cth_thresh_2 = np.zeros((ny,nx))
        ctt_thresh_2[:,:] = np.nan
        cth_thresh_2[:,:] = np.nan
    
    #--------------------------
    # Thresh 3 (Robust)
    #--------------------------
    thresh = 1.e-1
    cloud_id = np.where(qt > thresh)
    
    if np.size(cloud_id) > 0.:
        cloud_mask = np.zeros([nz,ny,nx])
        cloud_mask[cloud_id] = 1.
        ctt_thresh_3,cth_thresh_3 = calc_ctt(temp,z,qt,cloud_mask,nx,ny)
    else:
        ctt_thresh_3 = np.zeros((ny,nx))
        cth_thresh_3 = np.zeros((ny,nx))
        ctt_thresh_3[:,:] = np.nan
        cth_thresh_3[:,:] = np.nan

    print('Computing effective radii...')
    #===================================================================================
    # Calculate the effective radius of cloud liquid water, cloud ice, and snow
    # This is entirely consistent with how the Thompson microphysics scheme does it
    #===================================================================================
    re_cloud = np.zeros((nz,ny,nx))
    re_ice = np.zeros((nz,ny,nx))
    re_snow = np.zeros((nz,ny,nx))
    
    for iii in range(nx):
        for jjj in range(ny):
            t1d = temp[:,jjj,iii]+273.15 # K
            p1d = pressure[:,jjj,iii] # Pa
            qv1d = qv[:,jjj,iii]*1.e-3 # kg/kg
            qc1d = qc[:,jjj,iii]*1.e-3 # kg/kg
            nc1d = nc[:,jjj,iii] # /kg
            qi1d = qi[:,jjj,iii]*1.e-3 # kg/kg
            ni1d = ni[:,jjj,iii] # /kg
            qs1d = qs[:,jjj,iii]*1.e-3 # kg/kg
            re_qc1d = np.zeros_like(qc1d)
            re_qi1d = np.zeros_like(qc1d)
            re_qs1d = np.zeros_like(qc1d)
            kts = 1
            kte = nz
            re_qc1d, re_qi1d, re_qs1d = calc_effectRad(t1d, p1d, qv1d, qc1d, nc1d, qi1d, ni1d, qs1d, 
                                           re_qc1d, re_qi1d, re_qs1d, kts, kte, is_aerosol_aware=True)
    
            re_cloud[:,jjj,iii] = re_qc1d
            re_ice[:,jjj,iii] = re_qi1d
            re_snow[:,jjj,iii] = re_qs1d
    


    out_dict = {'re_cloud':re_cloud,\
                're_ice':re_ice,\
                're_snow':re_snow,\
                'z':z,\
                'qc':qc,\
                'qi':qi,\
                'qs':qs,\
                'temp':temp,\
                'ctt_thresh_1':ctt_thresh_1,\
                'cth_thresh_1':cth_thresh_1*1.e-3,\
                'ctt_thresh_2':ctt_thresh_2,\
                'cth_thresh_2':cth_thresh_2*1.e-3,\
                'ctt_thresh_3':ctt_thresh_3,\
                'cth_thresh_3':cth_thresh_3*1.e-3,\
    }

    return out_dict

In [ ]:
def compute_ctt_driver(in_dict,LUT,x2d_dict):

    re_cloud = in_dict['re_cloud'][1:-1,:,:]
    re_ice = in_dict['re_ice'][1:-1,:,:]
    re_snow = in_dict['re_snow'][1:-1,:,:]
    qc = in_dict['qc'][1:-1,:,:]
    qi = in_dict['qi'][1:-1,:,:]
    qs = in_dict['qs'][1:-1,:,:]
    temp = in_dict['temp']
    z = in_dict['z']


    #=====================================================
    # Compute CTT using an optical depth thresholding technique
    #=====================================================
    # nx = 86; ny = 111; nz = 40
    # temp (temperature); dimensions: (nz,ny,nx); units: deg C 
    # rho_air (moist air density); dimensions: (nz,ny,nz); units: kg/m^3
    # z (height above mean sea level); dimensions: (nz,ny,nx); units: m
    # qc (cloud water mass mixing ratio); dimensions: (nz,ny,nx); units: g/kg
    # qi (cloud ice mass mixing ratio); dimensions: (nz,ny,nx); units: g/kg
    # qs (snow mass mixing ratio); dimensions: (nz,ny,nx); units: g/kg
    # re_cloud (cloud water effective radius); dimensions: (nz,ny,nx); units: m
    # re_ice (cloud ice effective radius); dimensions: (nz,ny,nx); units: m
    # re_snow (cloud snow effective radius); dimensions: (nz,ny,nx); units: m

    #=======================
    # Compute dz
    #=======================
    dz = compute_dz(z)


    #=======================
    # Pack hydrometeor q's and r_eff for optical depth
    #=======================
    q_dict = {'cloud': qc*1e-3, 'ice': qi*1e-3, 'snow': qs*1e-3}  # kg/kg
    r_eff_dict = {'cloud': re_cloud, 'ice': re_ice, 'snow': re_snow} # meters

    #=======================
    # Hydrometeor densities
    #=======================
    DENSITIES = {'cloud': 1000.0, 'ice': 890.0, 'snow': 100.0}
    weightings = {
        '10_90': (0.1, 0.9),
        '30_70': (0.3, 0.7),
        '50_50': (0.5, 0.5),
        '70_30': (0.7, 0.3),
        '90_10': (0.9, 0.1)
    }

    #=======================
    # Compute optical depths using LUTs and weightings
    #=======================
    tau_VIS, tau_IR, tau_combined = compute_all_tau(q_dict, r_eff_dict, dz, 
                                                    DENSITIES=DENSITIES,
                                                    LUT=LUT,
                                                    weightings=weightings)

    tau_cum_dict = {}
    tau_cum_dict['tau_VIS'] = np.nansum(tau_VIS,axis=0)
    tau_cum_dict['tau_IR'] = np.nansum(tau_IR,axis=0)
    for key,val in tau_combined.items():
        tau_cum_dict['tau_'+key] = np.nansum(val,axis=0)
        
    #=======================
    # Compute cloud-top temperature/height from cumulative tau
    #=======================
    tau_dict_for_ctt = {"tau_VIS": tau_VIS, "tau_IR": tau_IR}
    #tau_dict_for_ctt.update(tau_combined)
    tau_dict_for_ctt['tau_combined'] = tau_combined

    
    ctt_cth_results = compute_ctt_cth_vectorized(tau_dict_for_ctt, temp[1:-1,:,:]+273.15, z[1:-1,:,:], tau_threshold=1.0)


    return tau_cum_dict, ctt_cth_results  

In [ ]:
def write_file(ref_dict,x2d_dict, x3d_dict, ctt_cth_dict,tau_cum_dict,out_datetime):
    out_dict = {'col_max_ref':ref_dict['col_max_ref'],
                'lwp':x2d_dict['lwp']*1.e3,
                'iwp':x2d_dict['iwp']*1.e3,
                'twp':x2d_dict['twp']*1.e3,
                'ctt_thresh_1':x3d_dict['ctt_thresh_1'],
                'ctt_thresh_2':x3d_dict['ctt_thresh_2'],
                'ctt_thresh_3':x3d_dict['ctt_thresh_3'],
                'cth_thresh_1':x3d_dict['cth_thresh_1'],
                'cth_thresh_2':x3d_dict['cth_thresh_2'],
                'cth_thresh_3':x3d_dict['cth_thresh_3'],
                'ctt_tau_vis':ctt_cth_dict['ctt_tau_VIS'],
                'cth_tau_vis':ctt_cth_dict['cth_tau_VIS'],
                'ctt_tau_ir':ctt_cth_dict['ctt_tau_IR'],
                'cth_tau_ir':ctt_cth_dict['cth_tau_IR'],
                'ctt_tau_10_90':ctt_cth_dict['ctt_tau_10_90'],
                'cth_tau_10_90':ctt_cth_dict['cth_tau_10_90'],
                'ctt_tau_30_70':ctt_cth_dict['ctt_tau_30_70'],
                'cth_tau_30_70':ctt_cth_dict['cth_tau_30_70'],
                'ctt_tau_50_50':ctt_cth_dict['ctt_tau_50_50'],
                'cth_tau_50_50':ctt_cth_dict['cth_tau_50_50'],
                'ctt_tau_70_30':ctt_cth_dict['ctt_tau_70_30'],
                'cth_tau_70_30':ctt_cth_dict['cth_tau_70_30'],
                'ctt_tau_90_10':ctt_cth_dict['ctt_tau_90_10'],
                'cth_tau_90_10':ctt_cth_dict['cth_tau_90_10'],
                'opd_vis':tau_cum_dict['tau_VIS'],
                'opd_ir':tau_cum_dict['tau_IR'],
                'opd_10_90':tau_cum_dict['tau_10_90'],
                'opd_30_70':tau_cum_dict['tau_30_70'],
                'opd_50_50':tau_cum_dict['tau_50_50'],
                'opd_70_30':tau_cum_dict['tau_70_30'],
                'opd_90_10':tau_cum_dict['tau_90_10'],
               }

    # Define variable attributes
    out_dict_attrs = {
                'col_max_ref': {
                    'long_name': 'Column-Maximum Radar Reflectivity',
                    'units': 'dBZ',
                },
                'twp':{
                    'long_name': 'Total Water Path',
                    'units':'g/m2'
                },
                'lwp':{
                    'long_name': 'Liquid Water Path',
                    'units':'g/m2'
                },
                'iwp':{
                    'long_name': 'Ice Water Path',
                    'units':'g/m2'
                },
                'opd_vis':{
                    'long_name': 'Visibile Optical Depth',
                    'units':'Dimensionless',
                    'description_1':'Optical depth calculated using cloud liquid water, cloud ice, and snow',
                    'description_2':'Corresponds to the visible wavelength of GOES-16 (0.64 um)',
                    'description_3':'The visible wavelength means that the scattering coefficient >> absorption coefficient in calculations',
                },
                'opd_ir':{
                    'long_name': 'Infrared Optical Depth',
                    'units':'Dimensionless',
                    'description_1':'Optical depth calculated using cloud liquid water, cloud ice, and snow',
                    'description_2':'Corresponds to the infrared (IR) wavelength of GOES-16 (11.2 um)',
                    'description_3':'The IR wavelength means that the scattering coefficient << absorption coefficient in calculations',
                },
                'opd_10_90':{
                    'long_name': 'Weighted (VIS:IR 10:90) Optical Depth',
                    'units':'Dimensionless',
                    'description_1':'Optical depth calculated using cloud liquid water, cloud ice, and snow',
                    'description_2':'Weighted between opd_vis and opd_ir in a 10:90 VIS:IR weighting',
                },
                'opd_30_70':{
                    'long_name': 'Weighted (VIS:IR 30:70) Optical Depth',
                    'units':'Dimensionless',
                    'description_1':'Optical depth calculated using cloud liquid water, cloud ice, and snow',
                    'description_2':'Weighted between opd_vis and opd_ir in a 30:70 VIS:IR weighting',
                },
                'opd_50_50':{
                    'long_name': 'Weighted (VIS:IR 50:50) Optical Depth',
                    'units':'Dimensionless',
                    'description_1':'Optical depth calculated using cloud liquid water, cloud ice, and snow',
                    'description_2':'Weighted between opd_vis and opd_ir in a 50:50 VIS:IR weighting',
                },
                'opd_70_30':{
                    'long_name': 'Weighted (VIS:IR 70:30) Optical Depth',
                    'units':'Dimensionless',
                    'description_1':'Optical depth calculated using cloud liquid water, cloud ice, and snow',
                    'description_2':'Weighted between opd_vis and opd_ir in a 70:30 VIS:IR weighting',
                },
                'opd_90_10':{
                    'long_name': 'Weighted (VIS:IR 90:10) Optical Depth',
                    'units':'Dimensionless',
                    'description_1':'Optical depth calculated using cloud liquid water, cloud ice, and snow',
                    'description_2':'Weighted between opd_vis and opd_ir in a 90:10 VIS:IR weighting',
                },
                'ctt_tau_vis':{
                    'long_name': 'Cloud Top Temperature (using VIS optical depth)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top temperature calculated as the temperature (height) from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed from the VIS optical depth only (opd_vis)',
                },
                'ctt_tau_ir':{
                    'long_name': 'Cloud Top Temperature (using IR optical depth)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top temperature calculated as the temperature (height) from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed from the IR optical depth only (opd_ir)',
                },
                'cth_tau_vis':{
                    'long_name': 'Cloud Top Height (using VIS optical depth)',
                    'units':'kilometers',
                    'description_1':'Cloud top height calculated as the height from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed from the VIS optical depth only (opd_vis)',
                    'description_3':'Height is altitude above ground level',
                },
                'cth_tau_ir':{
                    'long_name': 'Cloud Top Height (using IR optical depth)',
                    'units':'kilometers',
                    'description_1':'Cloud top height calculated as the height from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed from the IR optical depth only (opd_ir)',
                    'description_3':'Height is altitude above ground level',
                },
                'ctt_tau_10_90':{
                    'long_name': 'Cloud Top Temperature (using VIS-IR combined optical depth with a 10:90 VIS:IR weighting)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top temperature calculated as the temperature (height) from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_10_90',
                },
                'ctt_tau_30_70':{
                    'long_name': 'Cloud Top Temperature (using VIS-IR combined optical depth with a 30:70 VIS:IR weighting)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top temperature calculated as the temperature (height) from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_30_70',
                },
                'ctt_tau_50_50':{
                    'long_name': 'Cloud Top Temperature (using VIS-IR combined optical depth with a 50:50 VIS:IR weighting)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top temperature calculated as the temperature (height) from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_50_50',
                },
                'ctt_tau_70_30':{
                    'long_name': 'Cloud Top Temperature (using VIS-IR combined optical depth with a 70:30 VIS:IR weighting)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top temperature calculated as the temperature (height) from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_70_30',
                },
                'ctt_tau_90_10':{
                    'long_name': 'Cloud Top Temperature (using VIS-IR combined optical depth with a 90:10 VIS:IR weighting)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top temperature calculated as the temperature (height) from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_90_10',
                },
                'cth_tau_10_90':{
                    'long_name': 'Cloud Top Height (using VIS-IR combined optical depth with a 10:90 VIS:IR weighting)',
                    'units':'meters',
                    'description_1':'Cloud top height calculated as the height from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_10_90',
                    'description_3':'Height is altitude above ground level',
                },
                'cth_tau_30_70':{
                    'long_name': 'Cloud Top Height (using VIS-IR combined optical depth with a 30:70 VIS:IR weighting)',
                    'units':'meters',
                    'description_1':'Cloud top height calculated as the height from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_30_70',
                    'description_3':'Height is altitude above ground level',
                },
                'cth_tau_50_50':{
                    'long_name': 'Cloud Top Height (using VIS-IR combined optical depth with a 50:50 VIS:IR weighting)',
                    'units':'meters',
                    'description_1':'Cloud top height calculated as the height from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_50_50',
                    'description_3':'Height is altitude above ground level',
                },
                'cth_tau_70_30':{
                    'long_name': 'Cloud Top Height (using VIS-IR combined optical depth with a 70:30 VIS:IR weighting)',
                    'units':'meters',
                    'description_1':'Cloud top height calculated as the height from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_70_30',
                    'description_3':'Height is altitude above ground level',
                },
                'cth_tau_90_10':{
                    'long_name': 'Cloud Top Height (using VIS-IR combined optical depth with a 90:10 VIS:IR weighting)',
                    'units':'meters',
                    'description_1':'Cloud top height calculated as the height from domain top downwards where cumulative optical depth exceeds a value of 1',
                    'description_2':'Cumulative optical depth is computed using opd_90_10',
                    'description_3':'Height is altitude above ground level',
                },
                'ctt_thresh_1':{
                    'long_name': 'Cloud Top Temperature (lenient threshold)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top is defined using a total water mixing ratio of 1.e-6 g/kg',
                    'description_2':'Total water mixing ratio only considers cloud liquid water, cloud ice, and snow',
                    'description_3':'Cloud-top ID omits tenuous layers by requiring the layer to be at least 2 grid points thick',
                },
                'ctt_thresh_2':{
                    'long_name': 'Cloud Top Temperature (generous threshold)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top is defined using a total water mixing ratio of 1.e-3 g/kg',
                    'description_2':'Total water mixing ratio only considers cloud liquid water, cloud ice, and snow',
                    'description_3':'Cloud-top ID omits tenuous layers by requiring the layer to be at least 2 grid points thick',
                },
                'ctt_thresh_3':{
                    'long_name': 'Cloud Top Temperature (robust threshold)',
                    'units':'degrees Celsius',
                    'description_1':'Cloud top is defined using a total water mixing ratio of 1.e-1 g/kg',
                    'description_2':'Total water mixing ratio only considers cloud liquid water, cloud ice, and snow',
                    'description_3':'Cloud-top ID omits tenuous layers by requiring the layer to be at least 2 grid points thick',
                },
                'cth_thresh_1':{
                    'long_name': 'Cloud Top Height (lenient threshold)',
                    'units':'km',
                    'description_1':'Cloud top is defined using a total water mixing ratio of 1.e-6 g/kg',
                    'description_2':'Total water mixing ratio only considers cloud liquid water, cloud ice, and snow',
                    'description_3':'Cloud-top ID omits tenuous layers by requiring the layer to be at least 2 grid points thick',
                },
                'cth_thresh_2':{
                    'long_name': 'Cloud Top Height (generous threshold)',
                    'units':'km',
                    'description_1':'Cloud top is defined using a total water mixing ratio of 1.e-3 g/kg',
                    'description_2':'Total water mixing ratio only considers cloud liquid water, cloud ice, and snow',
                    'description_3':'Cloud-top ID omits tenuous layers by requiring the layer to be at least 2 grid points thick',
                },
                'cth_thresh_3':{
                    'long_name': 'Cloud Top Height (robust threshold)',
                    'units':'km',
                    'description_1':'Cloud top is defined using a total water mixing ratio of 1.e-1 g/kg',
                    'description_2':'Total water mixing ratio only considers cloud liquid water, cloud ice, and snow',
                    'description_3':'Cloud-top ID omits tenuous layers by requiring the layer to be at least 2 grid points thick',
                },
    }   

    date_str = out_datetime.strftime('%Y%m%d_%H:%M:%S')
    # Output parameters
    output_path = f'/pscratch/sd/m/mckenna/wrf_lasso/coarse_grained/{sim_date}/{ens}/{domain}/derived/'
    output_filename = f'{output_path}LASSO_CACTI_2.5km_derived_{date_str}_{ens}_{domain}.nc'

    # Define dimensions
    time_dimname = 'time'
    lon_dimname = 'west_east'
    lat_dimname = 'south_north'

    var_dict = {}
    # Define output variable dictionary
    for key, value in out_dict.items():
        if np.ndim(value) == 2.:
            var_dict[key] = ([lat_dimname,lon_dimname],value,out_dict_attrs[key])
            
    wrf_epoch = np.asarray([int(out_datetime.timestamp())])

    # Define coordinate attributes
    coord_attr_dict = {'time':{'long_name':'WRF Epoch time','units':'Epoch'},
                       'lon':{'long_name':'Longitude','units':'degrees'},
                       'lat':{'long_name':'Latitude','units':'degrees'},
                      }
    # Define coordinates
    coord_dict = {
            'time': ([time_dimname],wrf_epoch,coord_attr_dict['time']),
            'lon': ([lat_dimname,lon_dimname],ref_dict['lon'],coord_attr_dict['lon']),
            'lat': ([lat_dimname,lon_dimname],ref_dict['lat'],coord_attr_dict['lat']),
            }

    # Define global attributes
    gattr_dict = {
        'title':  f'WRF-LASSO derived 2.5-km variables for coarse-grained domain {domain}', \
        'description_1':  f'Date: {sim_date}', \
        'description_2':  f'Ensemble Member: {ens}', \
        'description_3':  f'Domain: {domain}', \
        'Institution': 'Pacific Northwest National Laboratoy', \
        'Contact': 'McKenna Stanford, mckenna.stnaford@pnnl.gov', \
        'Created_on':  time.ctime(time.time()), \
        #'reflectivity_source_path': wrf_path, \
        #'2d_source_path': wrf_2d_path, \
        'Date': sim_date, \
    }

    # Define xarray dataset
    dsout = xr.Dataset(var_dict, coords=coord_dict, attrs=gattr_dict)
    
    # Delete file if it already exists
    if os.path.isfile(output_filename):
        os.remove(output_filename)
    
    # Set encoding/compression for all variables
    comp = dict(zlib=True)
    encoding = {var: comp for var in dsout.data_vars}
    
    # Write to netcdf file
    dsout.to_netcdf(path=output_filename, mode="w",
                    format="NETCDF4", unlimited_dims=time_dimname, encoding=encoding)

    #return output_filename
    return dsout,out_dict

In [ ]:
def driver_func(cld_file,met_file,cld_datetime,lut):
    ref_dict = compute_col_max_ref(cld_file)
    x2d_dict = get_2d_vars(cld_file)
    x3d_dict = get_3d_vars(cld_file,met_file)
    tau_cum_dict, ctt_cth_dict = compute_ctt_driver(x3d_dict,lut,x2d_dict)
    outfile_name = write_file(ref_dict, x2d_dict, x3d_dict, ctt_cth_dict, tau_cum_dict,cld_datetime)
    print('Completed file:',outfile_name)

# Serial execution

In [ ]:
%%time
for ii in range(num_cld_files):
    print(cld_datetimes[ii])
    ref_dict = compute_col_max_ref(cld_files[ii],csapr_files[0])
    x2d_dict = get_2d_vars(cld_files[ii],csapr_files[0])
    x3d_dict = get_3d_vars(cld_files[ii],met_files[ii],csapr_files[0])    
    tau_cum_dict, ctt_cth_dict = compute_ctt_driver(x3d_dict,lut,x2d_dict)
    #outfile_name = write_file(ref_dict, x2d_dict, x3d_dict, ctt_cth_dict, tau_cum_dict,cld_datetimes[ii])
    dsout,out_dict = write_file(ref_dict, x2d_dict, x3d_dict, ctt_cth_dict, tau_cum_dict,cld_datetimes[ii])
    plot_maps(ref_dict,x3d_dict,ctt_cth_dict,tau_cum_dict)
    #print(aaaaaa)

In [ ]:
dsout

In [ ]:
for ii in range(8199,len(wrf_ref_datetimes)):
    outfile_name = driver_func(wrf_ref_files[ii],wrf_2d_files[ii],wrf_3d_files[ii],wrf_ref_datetimes[ii],wrf_2d_datetimes[ii],wrf_3d_datetimes[ii],lut)
    print(aaaaaa)

# Parallel Exectuion

In [ ]:
results = []
for ii in range(len(wrf_ref_datetimes)):
#for ii in range(8199,8203):
    result = dask.delayed(driver_func)(wrf_ref_files[ii],wrf_2d_files[ii],wrf_3d_files[ii],wrf_ref_datetimes[ii],wrf_2d_datetimes[ii],wrf_3d_datetimes[ii],lut)
    results.append(result)

In [ ]:
# Trigger dask computation
final_result = dask.compute(*results)
wait(final_result)